# Fine-tune ResNet-50 (M2)

**AIN 320 final project, Phase 1.** A standard deep CNN with ImageNet transfer learning.

This notebook writes out the training code of `src/train.py` cell by cell so you can **read, change and test** it:
hyperparameters, augmentation, the model, the training loop. Only the data plumbing (split lists, the dataset class
that reads the working copy) is imported from `src/`.

| | |
|---|---|
| Runtime | Colab **T4 GPU** (also runs locally on CPU for a quick try with the Dev tier) |
| Needs | the working copy from `00_build_working_copy.ipynb` |
| Disconnected or interrupted? | Run the cells again. The training cell resumes from the last finished epoch. |
| Output | `outputs/runs/<RUN_NAME>/`: `best.pt`, `last.pt`, `history.csv`, `summary.json`, `config.yaml`, figures |

Suggested use: run once with `TIER = "dev"` (a few minutes) to check everything works, then `TIER = "main"`.

## 1 · Settings

In [ ]:
MODEL = "resnet50"           # fixed for this notebook
TIER = "dev"                # "dev" = 3.2k train pages, to try things quickly | "main" = 32k, for reported results
RUN_NAME = f"{MODEL}_{TIER}"  # run folder name; 02_evaluate.ipynb finds the run by this name
FRESH = False               # True = move an existing run of this name aside and start over

REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # TODO: your team's GitHub repo (Colab only)
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ain320_doccls"

## 2 · Setup: code, data, paths
On Colab: mounts Drive, gets the repo, unpacks the working copy to local disk. Locally: uses `data/` and `outputs/`.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/final-project")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL} "{REPO_DIR}"
    else:
        !git -C "{REPO_DIR}" pull
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    DATA_DIR = Path("/content/data/working_copy")       # fast local disk
    OUTPUT_DIR = Path(DRIVE_ROOT) / "outputs"            # checkpoints survive disconnects
    !python -m src.data.local_copy unpack --src "{DRIVE_ROOT}/working_copy" --dst "{DATA_DIR}"
else:
    # Running locally: the notebook lives in notebooks/, the repo root is one level up.
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO_DIR)
    DATA_DIR = Path(os.environ.get("DOCCLS_DATA_DIR", REPO_DIR / "data" / "working_copy"))
    OUTPUT_DIR = Path(os.environ.get("DOCCLS_OUTPUT_DIR", REPO_DIR / "outputs"))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
SPLITS_DIR = Path(os.environ.get("DOCCLS_SPLITS_DIR", REPO_DIR / "splits"))
RUN_DIR = OUTPUT_DIR / "runs" / RUN_NAME

from src.data.labels import tier_split_lists
TRAIN_LIST, VAL_LIST = tier_split_lists(TIER)
!python -m src.data.local_copy verify --data-dir "{DATA_DIR}" --splits-dir "{SPLITS_DIR}" --lists {TRAIN_LIST} {VAL_LIST}
print("data   :", DATA_DIR)
print("run dir:", RUN_DIR)

## 3 · Hyperparameters

Defaults come from `configs/resnet50.yaml` (the values `src/train.py` uses). Change anything here to
experiment. The whole dict is saved with every checkpoint, so each run records exactly what it used.

In [ ]:
CONFIG = {
    "run_name": RUN_NAME,
    "seed": 42,  # fixed seed for sampling order, init and augmentation
    "deterministic": False,  # True = reproducible cuDNN kernels, but slower
    "model": {
        "name": 'resnet50',
        "num_classes": 16,
        "in_chans": 3,  # 1 = grayscale; 3 = grayscale copied to RGB for ImageNet weights
        "pretrained": True,  # ImageNet weights
    },
    "data": {
        "tier": TIER,
        "image_size": 224,  # pages are 256x256 in the working copy, resized to this
        "normalize": {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},  # "auto" = mean/std computed from the training tier
        "num_workers": 2 if IN_COLAB else 0,  # Colab has 2 vCPUs; 0 avoids Windows worker issues locally
    },
    "augment": {
        "enabled": True,
        "rotation_deg": 4,  # small rotations only: scans are slightly skewed, never upside down
        "translate": 0.03,  # fraction of the image
        "scale": 0.05,  # +/- zoom
        "brightness": 0.2,
        "contrast": 0.2,
        "blur_p": 0.3,  # probability of a light Gaussian blur
        "noise_p": 0.3,  # probability of pixel noise
        "noise_std": 0.03,
    },
    "train": {
        "epochs": 15,
        "batch_size": 64,
        "lr": 0.0001,  # backbone (or whole network for the custom CNN)
        "head_lr": 0.001,  # new 16-way classifier; None = same as lr
        "weight_decay": 0.01,
        "label_smoothing": 0.1,  # RVL-CDIP has ~8-12% label noise
        "warmup_epochs": 1,  # linear warm-up, then cosine decay to 0
        "amp": True,  # mixed precision on GPU
        "grad_clip": 1.0,
        "early_stopping_patience": 4,  # epochs without a val macro-F1 gain before stopping
        "log_every": 50,  # batches between progress lines
    },
    "paths": {"splits_dir": str(SPLITS_DIR), "data_dir": str(DATA_DIR), "output_dir": str(OUTPUT_DIR)},
}

## 4 · Imports, seeds, device

In [ ]:
import copy
import csv
import json
import math
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from torch import nn
from torchvision import transforms as T

# Data plumbing stays in src/ (split lists, the Dataset class that reads the working copy).
from src.data.dataset import DocumentImageDataset, compute_mean_std, load_manifest_paths, make_loader
from src.data.labels import CLASSES, INVOICE_IDX, NUM_CLASSES
from src.utils import environment_info


def set_seed(seed, deterministic=False):
    """Seed Python, NumPy and PyTorch. deterministic=False keeps cuDNN autotuning (faster)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


set_seed(CONFIG["seed"], CONFIG["deterministic"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = CONFIG["train"]["amp"] and DEVICE.type == "cuda"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", "| AMP:", USE_AMP)
if DEVICE.type != "cuda":
    print("No GPU: fine for a quick look, far too slow for real training (Colab: Runtime > Change runtime type > T4).")

## 5 · Preprocessing and augmentation

Training augmentations simulate scan variation only: small rotation and shift, slight zoom, brightness/contrast,
blur and noise, with white fill. **No flips and no large rotations**, because they would destroy the layout cues
(PRD §5.2). Validation pages are only resized and normalized.

In [ ]:
PAPER_WHITE = 255  # fill colour for pixels exposed by rotation/translation


class AddGaussianNoise:
    """Add pixel noise with standard deviation `std` to a [0, 1] tensor, then clip."""

    def __init__(self, std):
        self.std = std

    def __call__(self, tensor):
        return (tensor + torch.randn_like(tensor) * self.std).clamp_(0.0, 1.0)


class RepeatChannels:
    """Copy a 1-channel tensor to `n` channels (grayscale -> RGB for ImageNet models)."""

    def __init__(self, n):
        self.n = n

    def __call__(self, tensor):
        return tensor.expand(self.n, -1, -1) if self.n > 1 else tensor


def build_transforms(cfg, mean, std, train):
    """Resize -> (augment) -> tensor -> (noise) -> channels -> normalize. No flips (PRD 5.2)."""
    size, in_chans, aug = cfg["data"]["image_size"], cfg["model"]["in_chans"], cfg["augment"]
    steps = [T.Resize((size, size), antialias=True)]
    if train and aug["enabled"]:
        steps += [
            T.RandomAffine(degrees=aug["rotation_deg"], translate=(aug["translate"], aug["translate"]),
                           scale=(1 - aug["scale"], 1 + aug["scale"]), fill=PAPER_WHITE),
            T.ColorJitter(brightness=aug["brightness"], contrast=aug["contrast"]),
            T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=aug["blur_p"]),
        ]
    steps.append(T.ToTensor())
    if train and aug["enabled"] and aug["noise_p"] > 0:
        steps.append(T.RandomApply([AddGaussianNoise(aug["noise_std"])], p=aug["noise_p"]))
    steps += [RepeatChannels(in_chans), T.Normalize(mean, std)]
    return T.Compose(steps)

## 6 · Data

In [ ]:
available = load_manifest_paths(DATA_DIR)

# Normalization statistics. "auto" = computed once from the training tier and then stored in the
# checkpoint, so a resumed run (and the evaluation later) uses exactly the same values.
norm = CONFIG["data"]["normalize"]
if norm["mean"] == "auto":
    last = RUN_DIR / "last.pt"
    if last.exists() and not FRESH:
        norm = torch.load(last, map_location="cpu", weights_only=False)["normalize"]
    else:
        stats_ds = DocumentImageDataset.from_split(SPLITS_DIR / f"{TRAIN_LIST}.csv", DATA_DIR, available=available)
        mean, std = compute_mean_std(stats_ds, CONFIG["data"]["image_size"], seed=CONFIG["seed"])
        norm = {"mean": mean, "std": std}
    CONFIG["data"]["normalize"] = norm
MEAN, STD = list(norm["mean"]), list(norm["std"])
print("normalize mean", MEAN, "std", STD)

train_ds = DocumentImageDataset.from_split(SPLITS_DIR / f"{TRAIN_LIST}.csv", DATA_DIR,
                                           build_transforms(CONFIG, MEAN, STD, train=True), available)
val_ds = DocumentImageDataset.from_split(SPLITS_DIR / f"{VAL_LIST}.csv", DATA_DIR,
                                         build_transforms(CONFIG, MEAN, STD, train=False), available)
tcfg = CONFIG["train"]
workers = CONFIG["data"]["num_workers"]
train_loader = make_loader(train_ds, tcfg["batch_size"], shuffle=True, num_workers=workers, seed=CONFIG["seed"],
                           drop_last=len(train_ds) > tcfg["batch_size"])
val_loader = make_loader(val_ds, tcfg["batch_size"] * 2, shuffle=False, num_workers=workers, seed=CONFIG["seed"])
print(f"tier {TIER}: {len(train_ds)} train / {len(val_ds)} val pages, {len(train_loader)} batches per epoch")

### Check one batch (shape, value range, labels) and the augmentations by eye

In [ ]:
images, labels = next(iter(make_loader(train_ds, 16, shuffle=True, num_workers=0, seed=CONFIG["seed"])))
print("batch", tuple(images.shape), images.dtype, f"range [{images.min():.2f}, {images.max():.2f}]")
print("labels", labels.tolist())


def show(tensor):
    """Undo normalization of channel 0 for display."""
    return (tensor[0] * STD[0] + MEAN[0]).clamp(0, 1)


fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for img, lab, ax in zip(images, labels, axes.flat):
    ax.imshow(show(img), cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASSES[lab], fontsize=9)
    ax.axis("off")
plt.suptitle("one augmented training batch")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
axes[0].imshow(show(val_ds[0][0]), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("no augmentation")
for ax in axes[1:]:
    ax.imshow(show(train_ds[0][0]), cmap="gray", vmin=0, vmax=1)
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7 · Model: ResNet-50 (M2, ImageNet-pretrained, fine-tuned)

torchvision's ResNet-50 with ImageNet weights (`IMAGENET1K_V2`). Only the final layer is replaced: 2048 features to
16 classes. **Fine-tuning strategy: differential learning rates.** The whole network trains, the pretrained backbone
at `lr` (1e-4) and the new random head at `head_lr` (1e-3), so the head learns quickly without wrecking the
pretrained features. Grayscale pages are copied to 3 channels and normalized with ImageNet statistics.

In [ ]:
from torchvision.models import ResNet50_Weights, resnet50


def build_model():
    m = CONFIG["model"]
    net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if m["pretrained"] else None)
    net.fc = nn.Linear(net.fc.in_features, m["num_classes"])  # 2048 -> 16, freshly initialised
    return net


def get_head(model):
    return model.fc


net = build_model()
print(f"ResNet-50: {sum(p.numel() for p in net.parameters()) / 1e6:.2f}M parameters; head: {net.fc}")

In [ ]:
def get_param_groups(model, lr, head_lr, weight_decay):
    """AdamW groups: (backbone | head) x (decay | no decay). Biases and norm weights get no weight decay."""
    head_ids = {id(p) for p in get_head(model).parameters()}
    groups = {k: [] for k in ("backbone_decay", "backbone_no_decay", "head_decay", "head_no_decay")}
    for param in model.parameters():
        if param.requires_grad:
            part = "head" if id(param) in head_ids else "backbone"
            groups[f"{part}_{'no_decay' if param.ndim <= 1 else 'decay'}"].append(param)
    head_lr = lr if head_lr is None else head_lr
    return [{"name": k, "params": v, "lr": head_lr if k.startswith("head") else lr,
             "weight_decay": 0.0 if k.endswith("no_decay") else weight_decay} for k, v in groups.items() if v]


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

## 8 · Sanity check: overfit one batch

Before a long run, check that the model and loop can memorise 32 pages (loss towards 0, accuracy ≥ 0.95). If this
fails, something is wrong with the data, labels or model; do not start training.

In [ ]:
def overfit_one_batch(steps=200, batch_size=32):
    """A correct pipeline must memorise one fixed batch. Uses a *fresh* model, so nothing below is affected."""
    probe = build_model().to(DEVICE)
    unaugmented = copy.copy(val_ds)           # validation transforms ...
    unaugmented.samples = train_ds.samples    # ... on training images
    x, y = next(iter(make_loader(unaugmented, min(batch_size, len(unaugmented)), True, 0, CONFIG["seed"])))
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=tcfg["head_lr"] or tcfg["lr"], weight_decay=0.0)
    probe.train()
    for step in range(1, steps + 1):
        loss = nn.functional.cross_entropy(probe(x), y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step == 1 or step % 50 == 0:
            print(f"step {step:4d}  loss {loss.item():.4f}")
    probe.eval()
    with torch.no_grad():
        accuracy = (probe(x).argmax(1) == y).float().mean().item()
    print(f"batch accuracy {accuracy:.3f} ->", "OK" if accuracy >= 0.95 else "PROBLEM: cannot memorise one batch")
    del probe
    return accuracy


overfit_one_batch()

## 9 · Training functions

Straight from `src/train.py`: metrics (accuracy, **macro-F1**, invoice P/R/F1), linear warm-up plus cosine LR
schedule, one training epoch (mixed precision, gradient clipping), and validation.

In [ ]:
def classification_metrics(y_true, y_pred):
    """Accuracy, macro-F1 and invoice precision/recall/F1."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[INVOICE_IDX], zero_division=0)
    return {"acc": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)),
            "invoice_precision": float(p[0]), "invoice_recall": float(r[0]), "invoice_f1": float(f[0])}


def make_scheduler(optimizer, total_steps, warmup_steps):
    """Per-step LR multiplier: linear warm-up to 1, then cosine decay to 0."""
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def group_lr(optimizer, prefix):
    return next((g["lr"] for g in optimizer.param_groups if g["name"].startswith(prefix)), float("nan"))


def train_one_epoch(epoch):
    """One pass over the training set with AMP, gradient clipping and a per-step LR schedule."""
    model.train()
    total_loss, correct, seen, start = 0.0, 0, 0, time.time()
    for step, (x, y) in enumerate(train_loader, 1):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if tcfg["grad_clip"]:
            scaler.unscale_(optimizer)  # clip the true gradients, not the scaled ones
            nn.utils.clip_grad_norm_(model.parameters(), tcfg["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        seen += y.size(0)
        if tcfg["log_every"] and step % tcfg["log_every"] == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)}  loss {total_loss / seen:.4f}  "
                  f"acc {correct / seen:.3f}  {seen / (time.time() - start):.0f} img/s")
    return {"loss": total_loss / seen, "acc": correct / seen, "images_per_second": seen / (time.time() - start)}


@torch.no_grad()
def evaluate(net, loader):
    """Loss and metrics on a held-out loader. Returns (metrics, y_true, y_pred)."""
    net.eval()
    total_loss, seen, y_true, y_pred = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = net(x)
        total_loss += criterion(logits.float(), y).item() * y.size(0)
        seen += y.size(0)
        y_true.append(y.cpu().numpy())
        y_pred.append(logits.argmax(1).cpu().numpy())
    y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)
    return {**classification_metrics(y_true, y_pred), "loss": total_loss / seen}, y_true, y_pred


def atomic_save(obj, path):
    """torch.save to a temp file, then rename: a disconnect mid-write never corrupts the checkpoint."""
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

## 10 · Model, optimizer, schedule

AdamW with separate parameter groups for the backbone and the new classifier head (and no weight decay on biases and
normalization weights). Cross-entropy with label smoothing.

In [ ]:
if FRESH and RUN_DIR.exists():
    backup = RUN_DIR.with_name(f"{RUN_NAME}_old_{time.strftime('%Y%m%d_%H%M%S')}")
    shutil.move(str(RUN_DIR), str(backup))
    print("FRESH: previous run moved to", backup)
    FRESH = False  # only once, so re-running the cells below resumes instead of restarting again
RUN_DIR.mkdir(parents=True, exist_ok=True)
LAST, BEST = RUN_DIR / "last.pt", RUN_DIR / "best.pt"

model = build_model().to(DEVICE)
if DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
optimizer = torch.optim.AdamW(get_param_groups(model, tcfg["lr"], tcfg["head_lr"], tcfg["weight_decay"]))
scheduler = make_scheduler(optimizer, tcfg["epochs"] * len(train_loader), tcfg["warmup_epochs"] * len(train_loader))
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=tcfg["label_smoothing"])
N_PARAMS = count_parameters(model)
with open(RUN_DIR / "config.yaml", "w") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False)
print(f"{MODEL}: {N_PARAMS / 1e6:.2f}M parameters; checkpoints in {RUN_DIR}")
for g in optimizer.param_groups:
    print(f"  {g['name']:18s} lr {g['lr']:.0e}  weight decay {g['weight_decay']}  tensors {len(g['params'])}")

## 11 · Train

Validates after every epoch and stops early when validation macro-F1 stops improving. After every epoch:

* `last.pt`: model, optimizer, scheduler, AMP scaler, epoch, best score and history, for resuming
* `best.pt`: weights of the best epoch only (used for evaluation)

**Interrupted (Stop button, Colab disconnect)?** Re-run the cells from the top. This cell reloads `last.pt` and
continues from the last finished epoch. To try it, stop the cell after an epoch and run it again (PRD Step 3).

In [ ]:
HISTORY_FIELDS = ["epoch", "lr_backbone", "lr_head", "train_loss", "train_acc", "val_loss", "val_acc",
                  "val_macro_f1", "val_invoice_precision", "val_invoice_recall", "val_invoice_f1",
                  "epoch_seconds", "train_images_per_second"]


def restore_latest_checkpoint():
    """Load last.pt if it exists. Re-running this cell after an interruption therefore resumes cleanly."""
    state = {"epoch": 0, "best_score": -1.0, "best_epoch": 0, "bad_epochs": 0, "history": [], "finished": False}
    if LAST.exists():
        ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        state.update({k: ckpt[k] for k in state})
        print(f"resumed from {LAST}: {ckpt['epoch']} epochs done, best val macro-F1 {ckpt['best_score']:.4f}")
    return state


def build_summary(state, stop_reason):
    best_row = next((h for h in state["history"] if h["epoch"] == state["best_epoch"]), {})
    return {"run_name": RUN_NAME, "model": MODEL, "tier": TIER, "finished": state["finished"],
            "stop_reason": stop_reason, "epochs_completed": state["epoch"], "best_epoch": state["best_epoch"],
            "best_val_macro_f1": state["best_score"], "best_val_acc": best_row.get("val_acc"),
            "best_val_invoice_f1": best_row.get("val_invoice_f1"), "train_images": len(train_ds),
            "val_images": len(val_ds), "parameters": N_PARAMS, "best_checkpoint": str(BEST),
            "best_checkpoint_mb": round(BEST.stat().st_size / 1e6, 1) if BEST.exists() else None,
            "train_minutes": round(sum(h["epoch_seconds"] for h in state["history"]) / 60, 1),
            "normalize": {"mean": MEAN, "std": STD}, "environment": environment_info(), "trained_in": "notebook"}


state = restore_latest_checkpoint()
if state["finished"]:
    print("This run already finished. Set FRESH = True (and re-run from the settings cell) to train again.")
else:
    for epoch in range(state["epoch"] + 1, tcfg["epochs"] + 1):
        t0 = time.time()
        tr = train_one_epoch(epoch)
        va, _, _ = evaluate(model, val_loader)
        row = {"epoch": epoch, "lr_backbone": group_lr(optimizer, "backbone"), "lr_head": group_lr(optimizer, "head"),
               "train_loss": round(tr["loss"], 5), "train_acc": round(tr["acc"], 5),
               "val_loss": round(va["loss"], 5), "val_acc": round(va["acc"], 5), "val_macro_f1": round(va["macro_f1"], 5),
               "val_invoice_precision": round(va["invoice_precision"], 5),
               "val_invoice_recall": round(va["invoice_recall"], 5), "val_invoice_f1": round(va["invoice_f1"], 5),
               "epoch_seconds": round(time.time() - t0, 1), "train_images_per_second": round(tr["images_per_second"], 1)}
        state["history"].append(row)
        state["epoch"] = epoch

        improved = va["macro_f1"] > state["best_score"]
        if improved:  # best.pt: weights only (lean), same format as src/train.py
            state.update(best_score=va["macro_f1"], best_epoch=epoch, bad_epochs=0)
            atomic_save({"model": model.state_dict(), "epoch": epoch, "val_metrics": va, "config": CONFIG,
                         "normalize": {"mean": MEAN, "std": STD}, "classes": CLASSES}, BEST)
        else:
            state["bad_epochs"] += 1
        stop_reason = None
        if state["bad_epochs"] >= tcfg["early_stopping_patience"]:
            stop_reason = f"early stopping: no val macro-F1 gain for {state['bad_epochs']} epochs"
        elif epoch == tcfg["epochs"]:
            stop_reason = "reached max epochs"
        state["finished"] = stop_reason is not None
        summary = build_summary(state, stop_reason)

        # last.pt: everything needed to resume (a disconnect costs at most this one epoch)
        atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), **state,
                     "normalize": {"mean": MEAN, "std": STD}, "config": CONFIG, "summary": summary}, LAST)
        pd.DataFrame(state["history"], columns=HISTORY_FIELDS).to_csv(RUN_DIR / "history.csv", index=False)
        (RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
        print(f"epoch {epoch}/{tcfg['epochs']}  train loss {row['train_loss']:.4f} acc {row['train_acc']:.4f} | "
              f"val loss {row['val_loss']:.4f} acc {row['val_acc']:.4f} macro-F1 {row['val_macro_f1']:.4f} "
              f"invoice-F1 {row['val_invoice_f1']:.4f} | {row['epoch_seconds']:.0f}s" + ("  * best" if improved else ""))
        if state["finished"]:
            break

    if state["finished"]:
        index_row = {"finished_at": time.strftime("%Y-%m-%d %H:%M:%S"), "run_name": RUN_NAME, "model": MODEL,
                     "tier": TIER, "best_epoch": state["best_epoch"], "best_val_macro_f1": round(state["best_score"], 5),
                     "best_val_acc": summary["best_val_acc"], "best_val_invoice_f1": summary["best_val_invoice_f1"],
                     "train_minutes": summary["train_minutes"], "parameters": N_PARAMS, "lr": tcfg["lr"],
                     "head_lr": tcfg["head_lr"], "batch_size": tcfg["batch_size"], "epochs": tcfg["epochs"],
                     "seed": CONFIG["seed"], "checkpoint": str(BEST)}
        index_path = OUTPUT_DIR / "runs_index.csv"
        new_file = not index_path.exists()
        with open(index_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(index_row))
            if new_file:
                writer.writeheader()
            writer.writerow(index_row)
        print(f"done: {stop_reason}; best val macro-F1 {state['best_score']:.4f} at epoch {state['best_epoch']}")

## 12 · Results

In [ ]:
from matplotlib.ticker import MaxNLocator

history = pd.read_csv(RUN_DIR / "history.csv")
PAIR = ["#2a78d6", "#eb6834"]  # blue = first series, orange = second (project chart palette)
style = dict(x="epoch", marker="o", markersize=5, linewidth=2, color=PAIR)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
history.plot(y=["train_loss", "val_loss"], ax=axes[0], title="loss", **style)
history.plot(y=["train_acc", "val_acc"], ax=axes[1], title="accuracy", **style)
history.plot(y=["val_macro_f1", "val_invoice_f1"], ax=axes[2], title="validation F1", **style)
for ax in axes:
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.suptitle(RUN_NAME)
plt.tight_layout()
plt.savefig(RUN_DIR / "curves.png", dpi=150)
plt.show()
print(json.dumps({k: v for k, v in json.loads((RUN_DIR / "summary.json").read_text()).items()
                  if k not in ("environment", "normalize")}, indent=2))

### Best checkpoint on the validation set: confusion matrix and per-class F1 (weakest classes first)

In [ ]:
# Best checkpoint on the VALIDATION set (the test set is only used in 02_evaluate.ipynb, once).
from src.evaluate import per_class_rows, plot_confusion_matrix

best_model = build_model().to(DEVICE)
best_model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=False)["model"])
metrics, y_true, y_pred = evaluate(best_model, val_loader)
print(f"best epoch on val: accuracy {metrics['acc']:.4f}, macro-F1 {metrics['macro_f1']:.4f}, invoice F1 {metrics['invoice_f1']:.4f}")
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plot_confusion_matrix(cm, RUN_DIR / "val_confusion_matrix.png", f"{RUN_NAME}, validation set")
from IPython.display import Image as ShowImage, display
display(ShowImage(str(RUN_DIR / "val_confusion_matrix.png"), width=700))
pd.DataFrame(per_class_rows(y_true, y_pred)).sort_values("f1")

## Next

* **Evaluate on the test set** with `02_evaluate.ipynb`: make sure this run's name (`RUN_NAME`, e.g. `resnet50_main`)
  is in its `RUNS` list. Checkpoints from this notebook use the same format as `src/train.py`, so evaluation, Grad-CAM and the
  comparison tables work unchanged.
* For the report, keep the final settings reproducible: copy any hyperparameter you changed in the `CONFIG` cell into
  `configs/<model>.yaml`, so `python -m src.train` gives the same run (PRD §7.5: no logic only in notebooks).